In [7]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [8]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [9]:
def load_keywords(json_file=ROOT / "json/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [10]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [11]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
                else:
                    print(f"Error extrayendo ejercicios de archivo: {file.name}, página {idx+1}")
                    with open("errores.txt", "a") as f:
                        f.write(texto)
                        
    return exercises

In [30]:
def classify_exercise(statement: str, topic: str, keywords = None) -> str:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    topic_keywords = keywords[topic]["topic_kw"]
    pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
    if re.search(pattern, statement):
        return topic

In [13]:
def create_df_exercises(folder: Path) -> pd.DataFrame:
    exercises_list = []
    if not isinstance(folder, Path):
        raise ValueError("La función espera un objeto Path con el directorio a procesar.")
        if not folder.is_dir():
            raise ValueError("El objeto Path debe ser un directorio.")
    
    for file in folder.iterdir():
        if not file.suffix == ".pdf": 
            continue
        exams, statements = [], []
        #print("Processing file: ", file.name)
        exercises = extract_exercises(file)
        for ex in exercises:
            if ex:
                exercises_list.append({"exam" : ex[0], "statement": ex[1]})
    return pd.DataFrame(exercises_list)

Clasificar ejercicios de una carpeta

In [14]:
# # RENOMBRAR CARPETAS PARA QUE COINCIDAN CON CLAVES DEL JSON CON KEYWORDS
# import shutil

# acentos = {"á" : 'a', "é" : 'e', 'í' : 'i', 'ó' : 'o'}
# translation_table = str.maketrans(acentos)

# for folder in Path("temas").iterdir():
#     new_folder = folder.name.lower().replace(" ", "_")
#     new_folder = new_folder.translate(translation_table)
    
#     new_folder = Path("/".join(folder.parts[:-1]) + f"/{new_folder}")
#     shutil.move(folder, new_folder)

Ordenar directorios sin acentos:

In [ ]:
TEMAS = sorted(Path("temas").iterdir())

In [17]:
for idx, topic in enumerate([i.name for i in TEMAS]):
    print(f"{idx}: {topic}")

0: acido_base
1: cantidad_quimica
2: configuracion_electronica
3: enlace_quimico
4: equilibrio_quimico
5: formulacion
6: reactividad_organica
7: redox
8: solubilidad
9: termoquimica


## Clasificar carpeta de ejercicios

In [35]:
df = create_df_exercises(TEMAS[0])


Error extrayendo ejercicios de archivo: 2025 - Ácido Base.pdf, página 5
Error extrayendo ejercicios de archivo: 2009 - Ácido Base.pdf, página 3
Error extrayendo ejercicios de archivo: 2026 - Ácido Base.pdf, página 5


In [36]:
df.head()

,exam,statement
0,QUÍMICA. 2025. JUNIO. EJERCICIO 3B,Se preparan 250 mL de una disolución acuosa de...
1,QUÍMICA. 2025. RESERVA 1. EJERCICIO 2B,"a) Razone, según la teoría de Brönsted-Lowry, ..."
2,QUÍMICA. 2025. RESERVA 1. EJERCICIO 5,LA QUÍMICA DEL ÁCIDO ACETILSALICÍLICO\nEl ácid...
3,QUÍMICA. 2025. RESERVA 2. EJERCICIO 3B,En el laboratorio nos encontramos una botella ...
4,QUÍMICA. 2013. JUNIO. EJERCICIO 5. OPCIÓN B,Se disuelven 10 g de hidróxido de sodio en agu...


In [45]:
df["topic"] = df.statement.apply(classify_exercise, topic=TEMAS[0].name)
df = df.explode("topic")
df = df.sort_values("exam").reset_index(drop=True)
df = df.drop_duplicates()

In [46]:
df.isna().sum()

exam         0
statement    0
topic        2
dtype: int64

In [47]:
df[df.topic.isna()]

,exam,statement,topic
248,QUÍMICA. 2026. JULIO. EJERCICIO 5,,NaN
251,QUÍMICA. 2026. RESERVA 1. EJERCICIO 5,,NaN


In [48]:
for i in df[df.topic.isna()].statement:
    print(f"{i}\n\n")

In [49]:
df.topic.unique()

<StringArray>
['acido_base', nan]
Length: 2, dtype: str